# STOR 566 Group 9 - Facial-Expression Classification under Image Degradation

FER-2013 degradation experiments comparing three models: a baseline CNN trained from scratch (93,575 parameters), ImageNet-pretrained ResNet-18, and ImageNet-pretrained ViT-B/16 (`google/vit-base-patch16-224`), with the pretrained models fine-tuned end to end.

Each model runs under standard training and Gaussian-noise augmentation (6 configurations, 3 seeds = 18 final runs), plus 9 reference runs using clean-only validation checkpoint selection. Evaluation covers clean test images, 5 Gaussian noise levels, and 5 reduced-resolution levels (plus extended severities), reporting accuracy, macro-F1, per-class recall, and confusion matrices on validation and test.

To run on Colab, select a T4 GPU runtime and run all cells; outputs are saved to Google Drive under `MyDrive/stor566_group9_fer` and completed runs are skipped on restart. On Kaggle (T4 GPU with internet enabled), outputs are written to `/kaggle/working/stor566_group9_fer/`.

Learning rate, early stopping, and augmentation settings are chosen on validation images (clean + noisy only; reduced resolution is excluded) and saved to `selection_<arch>.json` before any final test evaluation. See `ferlib.ARCH_SETTINGS` for grids and budgets.

In [ ]:
# Experiment settings
ARCHS = ["cnn", "resnet18", "vit_b16"]
SEEDS = (0, 1, 2)
CSV_PATH = None        # optional local path to fer2013.csv / icml_face_data.csv
REQUIRE_DRIVE = True   # require Google Drive mount on Colab

In [ ]:
import os, sys, time, json, glob, shutil, tarfile, tempfile, zipfile, importlib

IN_COLAB = "google.colab" in sys.modules
OUT_DIR = None
if IN_COLAB:
    from google.colab import drive
    for attempt in range(1, 21):
        try:
            drive.mount("/content/drive")
            OUT_DIR = "/content/drive/MyDrive/stor566_group9_fer"
            break
        except Exception as e:
            print(f"[{attempt}/20] Drive mount failed ({type(e).__name__}), retrying...")
            time.sleep(5)
    if OUT_DIR is None and REQUIRE_DRIVE:
        raise RuntimeError("Google Drive mount failed; set REQUIRE_DRIVE = False to write locally.")
if OUT_DIR is None:
    OUT_DIR = "/kaggle/working/stor566_group9_fer" if os.path.isdir("/kaggle/working") else os.path.abspath("stor566_group9_fer")
os.makedirs(OUT_DIR, exist_ok=True)
PARTIAL_DIR = os.path.join(tempfile.gettempdir(), "fer_partial")
REUSE_DIR = os.path.join(tempfile.gettempdir(), "fer_tune_states")

import torch
print("torch", torch.__version__, "| results folder:", OUT_DIR)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("CUDA GPU not available.")

In [ ]:
%%writefile ferlib.py
"""
ferlib.py - STOR 566 Group 9
Shared data, model, training, and evaluation code for the FER-2013 degradation experiments.
"""
from __future__ import annotations

import hashlib
import json
import math
import os
import time
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# FER-2013 label order (index = value of the "emotion" column)
FER_LABELS = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
N_CLASSES = 7
IMG = 48

# --- Constants ---
# Primary levels come from the validation pilot sweep; extended levels are reported separately.
NOISE_PRIMARY = [0.025, 0.05, 0.10, 0.15, 0.20]
NOISE_EXTENDED = [0.30, 0.40, 0.50]
RES_PRIMARY = [40, 32, 24, 16, 8]
RES_EXTENDED = [4]

EVAL_NOISE_SEED = 566009          # fixed seed for eval noise
SPLIT_SEED_OFFSET = {"train": 0, "val": 100_000, "test": 200_000}

# Deduplicated training-split pixel mean/std in [0, 1] (used by BaselineCNN).
TRAIN_MEAN = 0.5060
TRAIN_STD = 0.2547

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Expected sizes and hashes after duplicate filtering.
EXPECTED_SIZES = {"train": 27418, "val": 3291, "test": 3268}
EXPECTED_FINGERPRINT = "0c9741cfaef7236e98ba237a0d28bbf5094f2700fce0ae926fc3ec1ccbc2013a"            # order-independent
EXPECTED_ROW_ORDER_FINGERPRINT = "7f7e95f72057c7d0d75da2aaca44ab56b16954bb0b755304e0821f052f7753ae"  # informational

PRETRAINED = True

# Per-architecture LR grids, augmentation grid, and training budgets.
AUG_GRID = [(0.5, 0.1), (0.5, 0.2), (0.8, 0.1), (0.8, 0.2)]
ARCH_SETTINGS = {
    "cnn": dict(lr_grid=[3e-4, 1e-3, 3e-3], aug_grid=AUG_GRID, max_epochs=150, patience=15, batch_size=128),
    "resnet18": dict(lr_grid=[3e-5, 1e-4, 3e-4], aug_grid=AUG_GRID, max_epochs=12, patience=3, batch_size=128),
    "vit_b16": dict(lr_grid=[1e-5, 2e-5, 5e-5], aug_grid=AUG_GRID, max_epochs=4, patience=4, batch_size=32,
                    extra=dict(optimizer="adamw", weight_decay=0.01, schedule="linear")),
}
VIT_CHECKPOINT = "google/vit-base-patch16-224"


# --- Data ---
SPLIT_OF = {"Training": "train", "PublicTest": "val", "PrivateTest": "test"}
PRIORITY = {"train": 0, "val": 1, "test": 2}


def read_fer_csv(path):
    """Read fer2013.csv (emotion,pixels,Usage) or icml_face_data.csv (emotion, Usage, pixels)."""
    import pandas as pd

    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    missing = {"emotion", "pixels", "Usage"} - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing columns {missing}; found {list(df.columns)}")
    y = df["emotion"].to_numpy(np.int64)
    usage = np.array([str(u).strip() for u in df["Usage"]])
    X = np.empty((len(df), IMG * IMG), np.uint8)
    for i, p in enumerate(df["pixels"].to_numpy()):
        X[i] = np.array(p.split(), dtype=np.uint8)
    return X, y, usage


def audit_and_dedup(X, y, usage):
    """Proposal rule:
    (1) remove every copy of a pixel-identical image whose copies carry conflicting labels;
    (2) for the remaining duplicate groups keep one copy, priority train > val > test
        (first occurrence inside the chosen split).
    Returns {split: (X uint8 (N,48,48), y int64, original_row_index)} and audit statistics.
    """
    split = np.array([SPLIT_OF[u] for u in usage])
    groups = {}
    for i in range(len(X)):
        groups.setdefault(X[i].tobytes(), []).append(i)
    keep = np.zeros(len(X), bool)
    st = dict(n_rows=int(len(X)), n_unique_images=len(groups), duplicate_groups=0,
              cross_split_duplicate_groups=0, conflicting_label_groups=0,
              rows_removed_conflicting=0, rows_removed_redundant=0)
    for idxs in groups.values():
        if len(idxs) == 1:
            keep[idxs[0]] = True
            continue
        st["duplicate_groups"] += 1
        if len({split[i] for i in idxs}) > 1:
            st["cross_split_duplicate_groups"] += 1
        if len({int(y[i]) for i in idxs}) > 1:
            st["conflicting_label_groups"] += 1
            st["rows_removed_conflicting"] += len(idxs)
            continue
        best = min(idxs, key=lambda i: (PRIORITY[split[i]], i))
        keep[best] = True
        st["rows_removed_redundant"] += len(idxs) - 1
    out = {}
    for s in ("train", "val", "test"):
        m = keep & (split == s)
        out[s] = (X[m].reshape(-1, IMG, IMG).copy(), y[m].astype(np.int64), np.nonzero(m)[0])
    st["raw_sizes"] = {s: int((split == s).sum()) for s in ("train", "val", "test")}
    st["final_sizes"] = {s: int(len(out[s][1])) for s in out}
    return out, st


def data_fingerprint(data):
    """Order-independent hash of the three splits (matches across fer2013.csv and icml_face_data.csv)."""
    h = hashlib.sha256()
    for s in ("train", "val", "test"):
        X, y = data[s][0], data[s][1]
        rows = sorted(hashlib.md5(np.ascontiguousarray(X[i]).tobytes() + bytes([int(y[i])])).digest()
                      for i in range(len(y)))
        h.update(s.encode())
        h.update(b"".join(rows))
    return h.hexdigest()


def row_order_fingerprint(data):
    """Order-dependent hash of the split arrays."""
    h = hashlib.sha256()
    for s in ("train", "val", "test"):
        h.update(np.ascontiguousarray(data[s][0]).tobytes())
        h.update(np.ascontiguousarray(data[s][1].astype(np.int64)).tobytes())
    return h.hexdigest()


def save_data(data, path):
    np.savez_compressed(path, **{f"{k}_{s}": v for s in data for k, v in zip(("X", "y", "idx"), data[s])})


def load_data(path):
    z = np.load(path)
    return {s: (z[f"X_{s}"], z[f"y_{s}"], z[f"idx_{s}"]) for s in ("train", "val", "test")}


def class_counts(y):
    return np.bincount(y, minlength=N_CLASSES)


# --- Degradations (48x48 images in [0, 1]) ---
def to_float(x_uint8):
    """uint8 (N,48,48) -> float32 tensor (N,1,48,48) in [0,1]."""
    return torch.from_numpy(np.ascontiguousarray(x_uint8)).float().div_(255.0).unsqueeze(1)


def gaussian_noise(x01, sigma, seed):
    """Zero-mean Gaussian noise with std sigma per pixel, clipped to [0, 1] (fixed RandomState seed)."""
    if sigma == 0:
        return x01.clone()
    rs = np.random.RandomState(seed % (2 ** 32))
    z = torch.from_numpy(rs.standard_normal(size=tuple(x01.shape)).astype(np.float32))
    return (x01 + float(sigma) * z).clamp_(0.0, 1.0)


def reduce_resolution(x01, s):
    """Bilinear downsampling with antialiasing to s x s, then bilinear resizing back to 48 x 48."""
    if s >= IMG:
        return x01.clone()
    lo = F.interpolate(x01, size=(s, s), mode="bilinear", antialias=True, align_corners=False)
    hi = F.interpolate(lo, size=(IMG, IMG), mode="bilinear", align_corners=False)
    return hi.clamp_(0.0, 1.0)


def noise_seed(split, sigma):
    return EVAL_NOISE_SEED + SPLIT_SEED_OFFSET[split] + int(round(sigma * 1000))


def cond_name(kind, level=None):
    if kind == "clean":
        return "clean"
    return f"noise_{level:g}" if kind == "noise" else f"res_{int(level)}"


def all_conditions(extended=True):
    conds = [("clean", None)]
    conds += [("noise", s) for s in NOISE_PRIMARY + (NOISE_EXTENDED if extended else [])]
    conds += [("res", s) for s in RES_PRIMARY + (RES_EXTENDED if extended else [])]
    return conds


def make_condition(x01, split, kind, level=None):
    if kind == "clean":
        return x01
    if kind == "noise":
        return gaussian_noise(x01, level, noise_seed(split, level))
    if kind == "res":
        return reduce_resolution(x01, int(level))
    raise ValueError(kind)


def mixed_noise_set(x01, split):
    """Noisy validation copy assigning each image evenly to one of the five primary noise levels."""
    n = x01.shape[0]
    rs = np.random.RandomState((EVAL_NOISE_SEED + SPLIT_SEED_OFFSET[split] + 7) % (2 ** 32))
    level_idx = rs.permutation(n) % len(NOISE_PRIMARY)
    sig = torch.tensor(np.array(NOISE_PRIMARY, np.float32)[level_idx]).view(n, 1, 1, 1)
    z = torch.from_numpy(rs.standard_normal(size=tuple(x01.shape)).astype(np.float32))
    return (x01 + sig * z).clamp_(0.0, 1.0)


# --- Models ---
class BaselineCNN(nn.Module):
    """Three 3x3 conv blocks (32, 64, 128), ReLU, 2x2 max pool, global average pool, and linear head (93,575 params)."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(128, N_CLASSES)

    def forward(self, x01):
        # Standardize inputs using the training-set mean and standard deviation.
        x = (x01 - TRAIN_MEAN) / TRAIN_STD
        return self.classifier(self.features(x).mean(dim=(2, 3)))


class ResNet18FER(nn.Module):
    """torchvision ResNet-18 (ImageNet-1K weights, 7-class head, full fine-tuning)."""

    def __init__(self, pretrained=True):
        super().__init__()
        from torchvision.models import ResNet18_Weights, resnet18

        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        net.fc = nn.Linear(net.fc.in_features, N_CLASSES)
        self.net = net
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x01):
        x = F.interpolate(x01, size=(224, 224), mode="bilinear", align_corners=False)
        x = (x.expand(-1, 3, -1, -1) - self.mean) / self.std
        return self.net(x)


class ViTB16FER(nn.Module):
    """ViT-B/16 (google/vit-base-patch16-224, 7-class head, full fine-tuning)."""

    def __init__(self, pretrained=True):
        super().__init__()
        from transformers import ViTConfig, ViTForImageClassification

        if pretrained:
            self.net = ViTForImageClassification.from_pretrained(VIT_CHECKPOINT, num_labels=N_CLASSES,
                                                                 ignore_mismatched_sizes=True)
        else:
            self.net = ViTForImageClassification(ViTConfig(num_labels=N_CLASSES))   # defaults = ViT-B/16, 224

    def forward(self, x01):
        x = F.interpolate(x01, size=(224, 224), mode="bilinear", align_corners=False)
        x = (x.expand(-1, 3, -1, -1) - 0.5) / 0.5
        return self.net(pixel_values=x).logits


def build_model(arch, pretrained=True):
    if arch == "cnn":
        return BaselineCNN()
    if arch == "resnet18":
        return ResNet18FER(pretrained=pretrained)
    if arch == "vit_b16":
        return ViTB16FER(pretrained=pretrained)
    raise ValueError(arch)


EVAL_BATCH = {"cnn": 512, "resnet18": 256, "vit_b16": 128}


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# --- Metrics ---
def compute_metrics(y, p):
    cm = np.zeros((N_CLASSES, N_CLASSES), np.int64)
    np.add.at(cm, (np.asarray(y), np.asarray(p)), 1)
    support, predicted, tp = cm.sum(1), cm.sum(0), np.diag(cm)
    recall = tp / np.maximum(support, 1)
    f1 = np.where(support + predicted > 0, 2 * tp / np.maximum(support + predicted, 1), 0.0)
    return dict(acc=float(tp.sum() / cm.sum()), macro_f1=float(f1.mean()),
                recall=[float(v) for v in recall], f1=[float(v) for v in f1], cm=cm.tolist())


@torch.no_grad()
def predict(model, x01, device, batch_size=512, amp=False):
    model.eval()
    out = []
    for i in range(0, x01.shape[0], batch_size):
        xb = x01[i:i + batch_size].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=amp):
            out.append(model(xb).argmax(1).cpu())
    return torch.cat(out).numpy()


def evaluate_split(model, data, split, device, conditions=None, batch_size=512, amp=False):
    """Metrics and predictions for one split under every condition."""
    conditions = conditions or all_conditions()
    x01, y = to_float(data[split][0]), data[split][1]
    res, preds = {}, {}
    for kind, level in conditions:
        name = cond_name(kind, level)
        p = predict(model, make_condition(x01, split, kind, level), device, batch_size, amp)
        res[name] = compute_metrics(y, p)
        preds[name] = p.astype(np.uint8)
    return res, preds


# --- Training ---
@dataclass
class TrainConfig:
    arch: str
    lr: float
    seed: int
    aug_p: float = 0.0           # probability that a training image receives Gaussian noise
    aug_sigma_max: float = 0.0   # for a noised image, sigma ~ Uniform(0, aug_sigma_max)
    batch_size: int = 128
    max_epochs: int = 60
    patience: int = 10
    select_on: str = "clean+noisy"   # "clean+noisy" or "clean"
    optimizer: str = "adam"          # "adam" or "adamw"
    weight_decay: float = 0.0
    schedule: str = "constant"       # "constant", or "linear": 10% linear warm-up then linear decay to zero

    @property
    def variant(self):
        if self.aug_p > 0:
            return "aug"
        return "ref" if self.select_on == "clean" else "std"

    def name(self):
        aug = f"-p{self.aug_p:g}-s{self.aug_sigma_max:g}" if self.aug_p > 0 else ""
        return f"{self.arch}-{self.variant}-lr{self.lr:g}{aug}-seed{self.seed}"


def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def selection_sets(data):
    """Validation tensors for LR, early stopping, and augmentation selection (clean + noisy only)."""
    xv = to_float(data["val"][0])
    rs = np.random.RandomState(EVAL_NOISE_SEED % (2 ** 32))
    sub = np.sort(rs.choice(len(data["train"][1]), size=len(data["val"][1]), replace=False))
    return dict(val_clean=xv, val_noisy=mixed_noise_set(xv, "val"), y_val=data["val"][1],
                train_sub=to_float(data["train"][0][sub]), y_train_sub=data["train"][1][sub])


def train_model(cfg: TrainConfig, data, device, sel=None, log=print, ckpt_path=None):
    """Train one model and return (best_state_dict_on_cpu, info). Saves per-epoch state if ckpt_path is set."""
    sel = sel or selection_sets(data)
    seed_everything(cfg.seed)
    model = build_model(cfg.arch, pretrained=PRETRAINED).to(device)
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
        torch.backends.cudnn.benchmark = True
    amp = device.type == "cuda"
    opt_cls = torch.optim.AdamW if cfg.optimizer == "adamw" else torch.optim.Adam
    opt = opt_cls(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    sched = None
    if cfg.schedule == "linear":
        total = cfg.max_epochs * math.ceil(len(data["train"][1]) / cfg.batch_size)
        warm = int(0.1 * total)
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda k: k / max(1, warm) if k < warm else max(0.0, (total - k) / max(1, total - warm)))

    xtr = torch.from_numpy(np.ascontiguousarray(data["train"][0])).to(device)
    ytr = torch.from_numpy(data["train"][1]).to(device)
    n = xtr.shape[0]
    g_order = torch.Generator().manual_seed(cfg.seed)                     # batch shuffle
    g_noise = torch.Generator(device=device).manual_seed(cfg.seed + 7919)  # training noise
    eval_bs = EVAL_BATCH[cfg.arch]

    best = dict(score=-1.0, epoch=0, state=None)
    history = []
    t0 = time.time()
    start_epoch, elapsed_before = 1, 0.0
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if ck["name"] == cfg.name():
            model.load_state_dict(ck["model"])
            opt.load_state_dict(ck["opt"])
            scaler.load_state_dict(ck["scaler"])
            if sched is not None and ck.get("sched") is not None:
                sched.load_state_dict(ck["sched"])
            g_order.set_state(ck["g_order"])
            g_noise.set_state(ck["g_noise"].to(device) if device.type == "cuda" else ck["g_noise"])
            best, history, elapsed_before = ck["best"], ck["history"], ck["elapsed"]
            start_epoch = len(history) + 1
            log(f"[{cfg.name()}] resumed after epoch {len(history)}")
    stopped = bool(history) and (history[-1]["epoch"] - best["epoch"] >= cfg.patience)
    for epoch in ([] if stopped else range(start_epoch, cfg.max_epochs + 1)):
        model.train()
        perm = torch.randperm(n, generator=g_order).to(device)
        loss_sum = torch.zeros((), device=device)
        correct = torch.zeros((), device=device)
        te = time.time()
        for i in range(0, n, cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            x = xtr[idx].float().div_(255.0).unsqueeze(1)
            y = ytr[idx]
            if cfg.aug_p > 0:
                b = x.shape[0]
                on = (torch.rand(b, generator=g_noise, device=device) < cfg.aug_p).float()
                sig = torch.rand(b, generator=g_noise, device=device) * cfg.aug_sigma_max
                z = torch.randn(x.shape, generator=g_noise, device=device)
                x = (x + (on * sig).view(b, 1, 1, 1) * z).clamp_(0.0, 1.0)
            with torch.autocast(device_type=device.type, enabled=amp):
                logits = model(x)
                loss = F.cross_entropy(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if sched is not None:
                sched.step()
            loss_sum += loss.detach().float() * len(idx)
            correct += (logits.argmax(1) == y).sum()
        acc_c = float((predict(model, sel["val_clean"], device, eval_bs, amp) == sel["y_val"]).mean())
        acc_n = float((predict(model, sel["val_noisy"], device, eval_bs, amp) == sel["y_val"]).mean())
        acc_t = float((predict(model, sel["train_sub"], device, eval_bs, amp) == sel["y_train_sub"]).mean())
        score = acc_c if cfg.select_on == "clean" else 0.5 * (acc_c + acc_n)
        history.append(dict(epoch=epoch, train_loss=float(loss_sum) / n, train_acc_running=float(correct) / n,
                            train_acc_eval=acc_t, val_acc_clean=acc_c, val_acc_noisy=acc_n, score=score,
                            epoch_seconds=round(time.time() - te, 1)))
        flag = ""
        if score > best["score"] + 1e-12:
            best = dict(score=score, epoch=epoch,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            flag = " *"
        log(f"[{cfg.name()}] ep {epoch:3d} loss {history[-1]['train_loss']:.4f} "
            f"train {acc_t:.4f} val {acc_c:.4f} val-noisy {acc_n:.4f} score {score:.4f} "
            f"({history[-1]['epoch_seconds']:.0f}s){flag}")
        if ckpt_path:
            tmp = ckpt_path + ".tmp"
            torch.save(dict(name=cfg.name(), model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                            sched=sched.state_dict() if sched is not None else None,
                            g_order=g_order.get_state(), g_noise=g_noise.get_state().cpu(), best=best, history=history,
                            elapsed=elapsed_before + time.time() - t0), tmp)
            os.replace(tmp, ckpt_path)
        if epoch - best["epoch"] >= cfg.patience:
            break
    info = dict(config=asdict(cfg), name=cfg.name(), variant=cfg.variant, best_epoch=best["epoch"],
                best_score=best["score"], epochs_run=len(history), history=history,
                train_seconds=round(elapsed_before + time.time() - t0, 1), n_params=count_params(model),
                device=str(device), torch_version=torch.__version__)
    return best["state"], info


# --- Experiment runner ---
class Runner:
    HEAVY = ("vit_b16",)   # skip per-epoch ckpts and reuse tuning weights for seed 0

    def __init__(self, out_dir, data, device, log_file=None, save_checkpoints=True,
                 partial_dir=None, reuse_dir=None):
        """partial_dir stores per-epoch resume checkpoints; reuse_dir caches HEAVY tuning weights for seed 0."""
        self.out, self.data, self.device = out_dir, data, device
        self.save_checkpoints = save_checkpoints
        self.partial_dir, self.reuse_dir = partial_dir, reuse_dir
        for d in ("tune", "final", "ckpt") + tuple(x for x in (partial_dir, reuse_dir) if x):
            os.makedirs(os.path.join(out_dir, d), exist_ok=True)
        self.log_file = log_file or os.path.join(out_dir, "log.txt")
        self.sel = selection_sets(data)
        self.fingerprint = data_fingerprint(data)

    def log(self, msg):
        line = f"{time.strftime('%H:%M:%S')} {msg}"
        print(line, flush=True)
        with open(self.log_file, "a") as f:
            f.write(line + "\n")

    def _path(self, stage, cfg):
        return os.path.join(self.out, stage, cfg.name() + ".json")

    def _train(self, stage, cfg):
        use_partial = self.partial_dir and cfg.arch not in self.HEAVY
        ck = os.path.join(self.partial_dir, f"{stage}-{cfg.name()}.pt") if use_partial else None
        state, info = train_model(cfg, self.data, self.device, self.sel, self.log, ck)
        return state, info, ck

    @staticmethod
    def _done(ck):
        if ck and os.path.exists(ck):
            try:
                os.remove(ck)
            except OSError:
                pass

    def tune_run(self, cfg):
        """Selection run: touches training and validation images only."""
        p = self._path("tune", cfg)
        if os.path.exists(p):
            return json.load(open(p))
        state, info, ck = self._train("tune", cfg)
        info["data_fingerprint"] = self.fingerprint
        if self.reuse_dir and cfg.arch in self.HEAVY:
            torch.save(state, self._reuse_path(cfg))
        json.dump(info, open(p, "w"))
        self._done(ck)
        return info

    def _reuse_path(self, cfg):
        return os.path.join(self.reuse_dir, cfg.name() + ".pt")

    def final_run(self, cfg):
        """Final run: trained, then evaluated on validation and test under every condition."""
        p = self._path("final", cfg)
        if os.path.exists(p):
            return json.load(open(p))
        tune_json = self._path("tune", cfg)
        if (self.reuse_dir and cfg.arch in self.HEAVY and os.path.exists(self._reuse_path(cfg))
                and os.path.exists(tune_json)):
            state, info, ck = torch.load(self._reuse_path(cfg), map_location="cpu"), json.load(open(tune_json)), None
            info["reused_tuning_run"] = True
            self.log(f"[{cfg.name()}] final run = selected tuning run (same configuration and seed), not retrained")
        else:
            state, info, ck = self._train("final", cfg)
        model = build_model(cfg.arch, pretrained=False).to(self.device)
        model.load_state_dict(state)
        amp = self.device.type == "cuda"
        bs = EVAL_BATCH[cfg.arch]
        info["metrics"], preds = {}, {}
        for split in ("val", "test"):
            info["metrics"][split], pr = evaluate_split(model, self.data, split, self.device, None, bs, amp)
            preds.update({f"{split}__{k}": v for k, v in pr.items()})
        info["data_fingerprint"] = self.fingerprint
        np.savez_compressed(os.path.join(self.out, "final", cfg.name() + "_preds.npz"), **preds)
        if self.save_checkpoints and cfg.arch not in self.HEAVY:
            try:
                torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in state.items()},
                           os.path.join(self.out, "ckpt", cfg.name() + ".pt"))
            except Exception as e:  # skip checkpoint if disk is full
                self.log(f"checkpoint not saved: {e}")
        json.dump(info, open(p, "w"))
        self._done(ck)
        m = info["metrics"]["test"]["clean"]
        self.log(f"FINAL {cfg.name()} best_epoch {info['best_epoch']} test acc {m['acc']:.4f} macro-F1 {m['macro_f1']:.4f}")
        return info

    def run_architecture(self, arch, lr_grid, aug_grid, seeds=(0, 1, 2), max_epochs=60, patience=10,
                         batch_size=128, extra=None):
        """Tune LR and augmentation on validation images (seed 0), save selection_<arch>.json, then run final seeds."""
        sel_path = os.path.join(self.out, f"selection_{arch}.json")
        base = dict(arch=arch, batch_size=batch_size, max_epochs=max_epochs, patience=patience, **(extra or {}))
        if os.path.exists(sel_path):
            sel = json.load(open(sel_path))
        else:
            lr_scores = {}
            for lr in lr_grid:
                lr_scores[lr] = self.tune_run(TrainConfig(lr=lr, seed=seeds[0], **base))["best_score"]
            best_lr = max(lr_grid, key=lambda v: (lr_scores[v], -lr_grid.index(v)))
            self.log(f"[{arch}] LR selection {lr_scores} -> {best_lr:g}")
            aug_scores = {}
            for p, s in aug_grid:
                r = self.tune_run(TrainConfig(lr=best_lr, seed=seeds[0], aug_p=p, aug_sigma_max=s, **base))
                aug_scores[f"{p:g}|{s:g}"] = r["best_score"]
            best_aug = max(aug_grid, key=lambda ps: (aug_scores[f"{ps[0]:g}|{ps[1]:g}"], -aug_grid.index(ps)))
            sel = dict(arch=arch, lr=best_lr, aug_p=best_aug[0], aug_sigma_max=best_aug[1],
                       lr_scores={f"{k:g}": v for k, v in lr_scores.items()}, aug_scores=aug_scores,
                       std_reference_score=lr_scores[best_lr], frozen_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                       noise_primary=NOISE_PRIMARY, res_primary=RES_PRIMARY)
            json.dump(sel, open(sel_path, "w"), indent=1)
            self.log(f"[{arch}] augmentation selection {aug_scores} -> p={best_aug[0]:g}, sigma_max={best_aug[1]:g}")
        results = []
        for seed in seeds:
            results.append(self.final_run(TrainConfig(lr=sel["lr"], seed=seed, **base)))
            results.append(self.final_run(TrainConfig(lr=sel["lr"], seed=seed, aug_p=sel["aug_p"],
                                                      aug_sigma_max=sel["aug_sigma_max"], **base)))
        return sel, results

    def run_reference(self, arch, lr_grid=None, aug_grid=None, seeds=(0, 1, 2), max_epochs=60, patience=10,
                      batch_size=128, extra=None):
        """Standard training at the selected LR with clean-only validation checkpoint selection. Reads selection_<arch>.json."""
        sel = json.load(open(os.path.join(self.out, f"selection_{arch}.json")))
        base = dict(arch=arch, batch_size=batch_size, max_epochs=max_epochs, patience=patience,
                    select_on="clean", **(extra or {}))
        return [self.final_run(TrainConfig(lr=sel["lr"], seed=seed, **base)) for seed in seeds]

In [ ]:
%%writefile analyze.py
"""Aggregate final runs into tables and figures.

Usage: python3 analyze.py <out_dir>          (reads <out_dir>/final/*.json, writes <out_dir>/analysis/)
Works on partial results (one architecture, fewer seeds).
"""
import glob
import json
import math
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import ferlib as fl

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

# ---- chart style (hue = architecture, line/marker style = training variant) ----
TEXT_DARK, TEXT_MID, TEXT_MUTED = "#111111", "#444444", "#777777"
GRID_COLOR, SPINE_COLOR, BG_COLOR = "#e5e5e5", "#bbbbbb", "#ffffff"
ARCH_COLOR = {"cnn": "#1f77b4", "resnet18": "#e0662b", "vit_b16": "#2ca02c"}
ARCH_LABEL = {"cnn": "Baseline CNN", "resnet18": "ResNet-18 (pretrained)", "vit_b16": "ViT-B/16 (pretrained)"}
VAR_LABEL = {"std": "standard training", "aug": "noise augmentation",
             "ref": "standard, clean-selected checkpoint"}
HEATMAP_CMAP = "Blues"
ARCHS, VARIANTS = ("cnn", "resnet18", "vit_b16"), ("std", "aug")
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11,
                     "axes.titleweight": "bold", "axes.titlelocation": "left", "figure.dpi": 100,
                     "savefig.dpi": 200, "savefig.facecolor": BG_COLOR, "figure.facecolor": BG_COLOR})


def style(ax, grid_axis="y"):
    ax.set_facecolor(BG_COLOR)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(SPINE_COLOR)
        ax.spines[s].set_linewidth(1)
    ax.grid(axis=grid_axis, color=GRID_COLOR, linewidth=1, linestyle="-")
    ax.set_axisbelow(True)
    ax.tick_params(colors=TEXT_MUTED, labelcolor=TEXT_MID, length=3)
    ax.title.set_color(TEXT_DARK)
    ax.xaxis.label.set_color(TEXT_MID)
    ax.yaxis.label.set_color(TEXT_MID)


def series_kw(arch, variant):
    c = ARCH_COLOR[arch]
    if variant == "std":
        return dict(color=c, linestyle="-", linewidth=2, marker="o", markersize=7,
                    markerfacecolor=c, markeredgecolor=BG_COLOR, markeredgewidth=1.5)
    if variant == "ref":
        return dict(color=c, linestyle=(0, (1, 1.5)), linewidth=2, marker="^", markersize=7,
                    markerfacecolor=c, markeredgecolor=BG_COLOR, markeredgewidth=1.5)
    return dict(color=c, linestyle=(0, (4, 2)), linewidth=2, marker="s", markersize=6.5,
                markerfacecolor=BG_COLOR, markeredgecolor=c, markeredgewidth=2)


# ---- loading and aggregation ----
def load_runs(out_dir):
    runs = {}
    for p in sorted(glob.glob(os.path.join(out_dir, "final", "*.json"))):
        r = json.load(open(p))
        runs[(r["config"]["arch"], r["variant"], r["config"]["seed"])] = r
    return runs


def seeds_of(runs, arch, variant):
    return sorted(s for (a, v, s) in runs if a == arch and v == variant)


def vals(runs, arch, variant, split, cond, metric="acc"):
    return np.array([runs[(arch, variant, s)]["metrics"][split][cond][metric] for s in seeds_of(runs, arch, variant)])


def mean_sd(x):
    x = np.asarray(x, float)
    return (float(x.mean()), float(x.std(ddof=1)) if len(x) > 1 else float("nan")) if len(x) else (float("nan"),) * 2


def fmt(x, pct=True, digits=1):
    m, s = mean_sd(x)
    k = 100 if pct else 1
    if np.isnan(m):
        return "n/a"
    return f"{m * k:.{digits}f}" + (f" ± {s * k:.{digits}f}" if not np.isnan(s) else "")


def conds(kind, extended=False):
    if kind == "noise":
        lv = fl.NOISE_PRIMARY + (fl.NOISE_EXTENDED if extended else [])
    else:
        lv = fl.RES_PRIMARY + (fl.RES_EXTENDED if extended else [])
    return [(lvl, fl.cond_name(kind, lvl)) for lvl in lv]


def configs(runs):
    return [(a, v) for a in ARCHS for v in VARIANTS if seeds_of(runs, a, v)]


# ---- tables ----
def write_tables(runs, out):
    rows = ["arch,variant,seed,split,condition,acc,macro_f1," + ",".join("recall_" + c for c in fl.FER_LABELS)]
    for (a, v, s), r in sorted(runs.items()):
        for split in ("val", "test"):
            for cname, m in r["metrics"][split].items():
                rows.append(f"{a},{v},{s},{split},{cname},{m['acc']:.5f},{m['macro_f1']:.5f}," +
                            ",".join(f"{x:.4f}" for x in m["recall"]))
    open(os.path.join(out, "summary_long.csv"), "w").write("\n".join(rows) + "\n")

    md = ["# Results summary (test split unless stated; mean ± sample SD over seeds, in %)", ""]
    md += ["## RQ1 - original images", "",
           "| Model | Training | Seeds | Val acc | Test acc | Test macro-F1 | Best epoch |", "|---|---|---|---|---|---|---|"]
    for a, v in configs(runs):
        be = [runs[(a, v, s)]["best_epoch"] for s in seeds_of(runs, a, v)]
        md.append(f"| {ARCH_LABEL[a]} | {VAR_LABEL[v]} | {len(be)} | {fmt(vals(runs, a, v, 'val', 'clean'))} | "
                  f"{fmt(vals(runs, a, v, 'test', 'clean'))} | {fmt(vals(runs, a, v, 'test', 'clean', 'macro_f1'))} | "
                  f"{', '.join(map(str, be))} |")
    for kind, title, unit in (("noise", "Gaussian noise", "sigma"), ("res", "reduced resolution", "size")):
        for metric, mname in (("acc", "accuracy"), ("macro_f1", "macro-F1")):
            cl = conds(kind, extended=True)
            md += ["", f"## RQ2 - test {mname} under {title} ({unit}; primary levels first, then extended)", "",
                   "| Model | Training | clean | " + " | ".join(f"{l:g}" for l, _ in cl) + " |",
                   "|---|---|---|" + "---|" * len(cl)]
            for a, v in configs(runs):
                md.append(f"| {ARCH_LABEL[a]} | {VAR_LABEL[v]} | {fmt(vals(runs, a, v, 'test', 'clean', metric))} | " +
                          " | ".join(fmt(vals(runs, a, v, "test", c, metric)) for _, c in cl) + " |")
        cl = conds(kind)
        md += ["", f"### Relative accuracy under {title} (degraded / original, per seed) and 90% threshold", "",
               "| Model | Training | " + " | ".join(f"{l:g}" for l, _ in cl) + " | first primary level below 90% |",
               "|---|---|" + "---|" * (len(cl) + 1)]
        for a, v in configs(runs):
            base = vals(runs, a, v, "test", "clean")
            rel = [vals(runs, a, v, "test", c) / base for _, c in cl]
            first = next((f"{l:g}" for (l, _), r in zip(cl, rel) if r.mean() < 0.90), "none in tested range")
            md.append(f"| {ARCH_LABEL[a]} | {VAR_LABEL[v]} | " + " | ".join(fmt(r) for r in rel) + f" | {first} |")
    md += ["", "## RQ3 - effect of noise augmentation (augmented minus standard, same seed; percentage points)", ""]
    cl = [(None, "clean")] + conds("noise") + conds("res")
    md += ["| Model | clean | " + " | ".join(c for _, c in cl[1:]) + " |", "|---|" + "---|" * len(cl)]
    for a in ARCHS:
        common = sorted(set(seeds_of(runs, a, "std")) & set(seeds_of(runs, a, "aug")))
        if not common:
            continue
        cells = []
        for _, c in cl:
            d = [runs[(a, "aug", s)]["metrics"]["test"][c]["acc"] - runs[(a, "std", s)]["metrics"]["test"][c]["acc"]
                 for s in common]
            m, sd = mean_sd(d)
            cells.append(f"{m * 100:+.1f}" + (f" ± {sd * 100:.1f}" if not np.isnan(sd) else ""))
        md.append(f"| {ARCH_LABEL[a]} | " + " | ".join(cells) + " |")
    ref_archs = [a for a in ARCHS if seeds_of(runs, a, "ref")]
    if ref_archs:
        cl = [(None, "clean")] + conds("noise") + conds("res")
        md += ["", "## Reference - standard training with the checkpoint chosen on clean validation accuracy only", "",
               "Same training and learning rate as 'standard training'; only the stopping rule differs.", "",
               "| Model | Seeds | Val acc | Test acc | Test macro-F1 | Best epoch | " + " | ".join(c for _, c in cl[1:]) + " |",
               "|---|---|---|---|---|---|" + "---|" * (len(cl) - 1)]
        for a in ref_archs:
            be = [runs[(a, "ref", s)]["best_epoch"] for s in seeds_of(runs, a, "ref")]
            md.append(f"| {ARCH_LABEL[a]} | {len(be)} | {fmt(vals(runs, a, 'ref', 'val', 'clean'))} | "
                      f"{fmt(vals(runs, a, 'ref', 'test', 'clean'))} | {fmt(vals(runs, a, 'ref', 'test', 'clean', 'macro_f1'))} | "
                      f"{', '.join(map(str, be))} | " + " | ".join(fmt(vals(runs, a, "ref", "test", c)) for _, c in cl[1:]) + " |")
        md += ["", "### Effect of the selection rule (clean + noisy selection minus clean-only selection, same seed; "
                   "percentage points of test accuracy)", "",
               "| Model | clean | " + " | ".join(c for _, c in cl[1:]) + " |", "|---|" + "---|" * len(cl)]
        for a in ref_archs:
            common = sorted(set(seeds_of(runs, a, "std")) & set(seeds_of(runs, a, "ref")))
            if not common:
                continue
            cells = []
            for _, c in cl:
                d = [runs[(a, "std", s)]["metrics"]["test"][c]["acc"] - runs[(a, "ref", s)]["metrics"]["test"][c]["acc"]
                     for s in common]
                m, sd = mean_sd(d)
                cells.append(f"{m * 100:+.1f}" + (f" ± {sd * 100:.1f}" if not np.isnan(sd) else ""))
            md.append(f"| {ARCH_LABEL[a]} | " + " | ".join(cells) + " |")
    md += ["", "## Per-class recall on original test images (%)", "",
           "| Model | Training | " + " | ".join(fl.FER_LABELS) + " |", "|---|---|" + "---|" * 7]
    for a, v in configs(runs):
        rec = np.array([runs[(a, v, s)]["metrics"]["test"]["clean"]["recall"] for s in seeds_of(runs, a, v)])
        md.append(f"| {ARCH_LABEL[a]} | {VAR_LABEL[v]} | " + " | ".join(f"{x * 100:.1f}" for x in rec.mean(0)) + " |")
    sel = {}
    for a in ARCHS:
        p = os.path.join(os.path.dirname(out), f"selection_{a}.json")
        if os.path.exists(p):
            sel[a] = json.load(open(p))
    if sel:
        md += ["", "## Selected hyperparameters (validation selection score only)", ""]
        for a, s in sel.items():
            md.append(f"- {ARCH_LABEL[a]}: lr {s['lr']:g} (scores {s['lr_scores']}); augmentation p={s['aug_p']:g}, "
                      f"sigma_max={s['aug_sigma_max']:g} (scores {s['aug_scores']}); frozen {s['frozen_at']}")
    md += ["", "SD values describe variation across training seeds; they are not confidence intervals."]
    open(os.path.join(out, "results_summary.md"), "w").write("\n".join(md) + "\n")
    return "\n".join(md)


# ---- figures ----
def majority_rate(runs, split="test"):
    r = next(iter(runs.values()))
    cm = np.array(r["metrics"][split]["clean"]["cm"])
    return float(cm.sum(1).max() / cm.sum())


def fig_degradation(runs, out, metric="acc", relative=False, extended=False, name="fig_degradation_acc"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharey=True)
    for ax, kind in zip(axes, ("noise", "res")):
        cl = conds(kind, extended)
        xs = [0.0 if kind == "noise" else 48] + [l for l, _ in cl]
        for a, v in configs(runs):
            base = vals(runs, a, v, "test", "clean", metric)
            ys = [base] + [vals(runs, a, v, "test", c, metric) for _, c in cl]
            if relative:
                ys = [y / base for y in ys]
            m = np.array([y.mean() for y in ys])
            sd = np.array([y.std(ddof=1) if len(y) > 1 else 0.0 for y in ys])
            ax.fill_between(xs, m - sd, m + sd, color=ARCH_COLOR[a], alpha=0.10, linewidth=0)
            ax.plot(xs, m, label=f"{ARCH_LABEL[a]}, {VAR_LABEL[v]}", **series_kw(a, v))
        style(ax)
        if kind == "res":
            ax.invert_xaxis()
            ax.set_xticks(xs)
            ax.set_xlabel("Intermediate image size (pixels per side; 48 = original)")
            ax.set_title("Reduced resolution")
        else:
            ax.set_xticks(xs)
            ax.set_xticklabels([f"{x:g}" for x in xs], rotation=0 if not extended else 45)
            ax.set_xlabel(r"Gaussian noise standard deviation $\sigma$ (pixels scaled to [0, 1])")
            ax.set_title("Gaussian noise")
        if relative:
            ax.axhline(0.90, color=TEXT_MUTED, linewidth=1, linestyle=(0, (2, 2)))
            ax.annotate("90% of original accuracy", xy=(xs[-1], 0.90), xytext=(0, 4), textcoords="offset points",
                        ha="right", va="bottom", fontsize=8.5, color=TEXT_MID)
        elif metric == "acc":
            mr = majority_rate(runs)
            ax.axhline(mr, color=TEXT_MUTED, linewidth=1, linestyle=(0, (2, 2)))
            ax.annotate(f"majority-class rate {mr * 100:.1f}%", xy=(xs[0], mr), xytext=(0, 4),
                        textcoords="offset points", ha="left", va="bottom", fontsize=8.5, color=TEXT_MID)
    ylab = {"acc": "Test accuracy", "macro_f1": "Test macro-F1"}[metric]
    axes[0].set_ylabel("Relative accuracy (degraded / original)" if relative else ylab)
    axes[0].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
    h, l = axes[0].get_legend_handles_labels()
    rows = math.ceil(len(l) / 2)
    order = sorted(range(len(l)), key=lambda i: (i % 2, i))   # legend fills by column: standard runs left, augmented right
    fig.legend([h[i] for i in order], [l[i] for i in order], frameon=False, fontsize=9, labelcolor=TEXT_MID, ncol=2,
               loc="upper center", bbox_to_anchor=(0.5, 1.0), handlelength=3, columnspacing=3)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.045 * rows - 0.03))
    fig.savefig(os.path.join(out, name + ".png"))
    plt.close(fig)


def fig_by_architecture(runs, out, metric="acc"):
    archs = [a for a in ARCHS if any(seeds_of(runs, a, v) for v in ("ref", "std", "aug"))]
    if not archs:
        return
    fig, axes = plt.subplots(2, len(archs), figsize=(4.4 * len(archs), 7.2), sharey=True, squeeze=False)
    for j, a in enumerate(archs):
        for i, kind in enumerate(("noise", "res")):
            ax = axes[i, j]
            cl = conds(kind)
            xs = [0.0 if kind == "noise" else 48] + [l for l, _ in cl]
            for v in ("ref", "std", "aug"):
                if not seeds_of(runs, a, v):
                    continue
                ys = [vals(runs, a, v, "test", "clean", metric)] + [vals(runs, a, v, "test", c, metric) for _, c in cl]
                m = np.array([y.mean() for y in ys])
                sd = np.array([y.std(ddof=1) if len(y) > 1 else 0.0 for y in ys])
                ax.fill_between(xs, m - sd, m + sd, color=ARCH_COLOR[a], alpha=0.10, linewidth=0)
                ax.plot(xs, m, label=VAR_LABEL[v], **series_kw(a, v))
            style(ax)
            ax.set_xticks(xs)
            if kind == "res":
                ax.invert_xaxis()
                ax.set_xlabel("Intermediate image size (px)")
            else:
                ax.set_xticklabels([f"{x:g}" for x in xs], fontsize=8.5)
                ax.set_xlabel(r"Gaussian noise $\sigma$")
                ax.set_title(ARCH_LABEL[a])
            mr = majority_rate(runs)
            ax.axhline(mr, color=TEXT_MUTED, linewidth=1, linestyle=(0, (2, 2)))
            if j == 0:
                ax.set_ylabel("Test accuracy")
                ax.annotate(f"majority-class rate {mr * 100:.1f}%", xy=(xs[0], mr), xytext=(0, 4),
                            textcoords="offset points", ha="left", va="bottom", fontsize=8, color=TEXT_MID)
            ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
    from matplotlib.lines import Line2D
    present = [v for v in ("ref", "std", "aug") if any(seeds_of(runs, a, v) for a in archs)]
    proxies = []
    for v in present:
        kw = series_kw(archs[0], v)
        kw.update(color=TEXT_MID)
        kw["markerfacecolor"] = BG_COLOR if v == "aug" else TEXT_MID
        kw["markeredgecolor"] = TEXT_MID if v == "aug" else BG_COLOR
        proxies.append(Line2D([], [], **kw))
    fig.legend(proxies, [VAR_LABEL[v] for v in present], frameon=False, fontsize=9, labelcolor=TEXT_MID, ncol=len(present),
               loc="upper center", bbox_to_anchor=(0.5, 1.0), handlelength=3.4, columnspacing=3)
    fig.tight_layout(rect=(0, 0, 1, 0.955))
    fig.savefig(os.path.join(out, "fig_by_architecture.png"))
    plt.close(fig)


def fig_aug_delta(runs, out):
    cl = [(None, "clean")] + conds("noise") + conds("res")
    labels = ["original"] + [rf"$\sigma$={l:g}" for l, _ in conds("noise")] + [f"{l}px" for l, _ in conds("res")]
    archs = [a for a in ARCHS if set(seeds_of(runs, a, "std")) & set(seeds_of(runs, a, "aug"))]
    if not archs:
        return
    fig, ax = plt.subplots(figsize=(11, 4.2))
    w = 0.78 / len(archs) - 0.04
    for k, a in enumerate(archs):
        common = sorted(set(seeds_of(runs, a, "std")) & set(seeds_of(runs, a, "aug")))
        d = np.array([[runs[(a, "aug", s)]["metrics"]["test"][c]["acc"] - runs[(a, "std", s)]["metrics"]["test"][c]["acc"]
                       for _, c in cl] for s in common]) * 100
        m = d.mean(0)
        sd = d.std(0, ddof=1) if len(common) > 1 else np.zeros_like(m)
        x = np.arange(len(cl)) + (k - (len(archs) - 1) / 2) * (w + 0.04)
        ax.bar(x, m, width=w, color=ARCH_COLOR[a], label=ARCH_LABEL[a], linewidth=0)
        ax.errorbar(x, m, yerr=sd, fmt="none", ecolor=TEXT_MID, elinewidth=1, capsize=2)
    style(ax)
    ax.axhline(0, color=SPINE_COLOR, linewidth=1)
    for xsep in (0.5, 0.5 + len(conds("noise"))):
        ax.axvline(xsep, color=GRID_COLOR, linewidth=1)
    ax.set_xticks(np.arange(len(cl)))
    ax.set_xticklabels(labels)
    ax.set_ylabel("Change in test accuracy (percentage points)")
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi + 0.14 * (hi - lo))
    ymax = ax.get_ylim()[1]
    ax.annotate("Gaussian noise (corruption seen in training)", xy=(3, ymax), xytext=(0, -3), textcoords="offset points",
                ha="center", va="top", fontsize=8.5, color=TEXT_MID)
    ax.annotate("Reduced resolution (unseen corruption)", xy=(3 + len(conds("noise")), ymax), xytext=(0, -3),
                textcoords="offset points", ha="center", va="top", fontsize=8.5, color=TEXT_MID)
    fig.suptitle("Effect of noise augmentation: augmented minus standard training (same seed)", x=0.01, y=0.985,
                 ha="left", fontsize=11, fontweight="bold", color=TEXT_DARK)
    h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, frameon=False, fontsize=9, labelcolor=TEXT_MID, ncol=3, loc="upper left", bbox_to_anchor=(0.045, 0.93))
    fig.tight_layout(rect=(0, 0, 1, 0.86))
    fig.savefig(os.path.join(out, "fig_augmentation_effect.png"))
    plt.close(fig)


def _heat(ax, M, xt, yt, title, vmax=1.0):
    cmap = plt.get_cmap(HEATMAP_CMAP)
    ax.imshow(M, cmap=cmap, vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(xt)))
    ax.set_xticklabels(xt, fontsize=8.5)
    ax.set_yticks(range(len(yt)))
    ax.set_yticklabels(yt, fontsize=9)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j] * 100:.0f}", ha="center", va="center", fontsize=8,
                    color="#ffffff" if M[i, j] > 0.55 * vmax else TEXT_DARK)
    ax.set_title(title, fontsize=10)
    ax.tick_params(colors=TEXT_MUTED, labelcolor=TEXT_MID, length=0)
    for s in ax.spines.values():
        s.set_visible(False)


def fig_recall_heatmaps(runs, out):
    cfgs = configs(runs)
    if not cfgs:
        return
    cl = [(None, "clean")] + conds("noise") + conds("res")
    xt = ["orig."] + [rf"$\sigma${l:g}" for l, _ in conds("noise")] + [f"{l}px" for l, _ in conds("res")]
    fig, axes = plt.subplots(len(cfgs), 1, figsize=(9.5, 2.5 * len(cfgs)), squeeze=False)
    for ax, (a, v) in zip(axes[:, 0], cfgs):
        M = np.array([np.mean([runs[(a, v, s)]["metrics"]["test"][c]["recall"] for s in seeds_of(runs, a, v)], 0)
                      for _, c in cl]).T
        _heat(ax, M, xt, fl.FER_LABELS, f"{ARCH_LABEL[a]}, {VAR_LABEL[v]} - per-class recall on test (%)")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig_per_class_recall.png"))
    plt.close(fig)


def fig_confusion(runs, out, variant="std"):
    archs = [a for a in ARCHS if seeds_of(runs, a, variant)]
    if not archs:
        return
    fig, axes = plt.subplots(1, len(archs), figsize=(5.6 * len(archs), 4.8), squeeze=False)
    for ax, a in zip(axes[0], archs):
        cm = sum(np.array(runs[(a, variant, s)]["metrics"]["test"]["clean"]["cm"]) for s in seeds_of(runs, a, variant))
        _heat(ax, cm / cm.sum(1, keepdims=True), fl.FER_LABELS, fl.FER_LABELS,
              f"{ARCH_LABEL[a]}, {VAR_LABEL[variant]} - original test images\nrow-normalized confusion (%), seeds pooled")
        ax.set_xlabel("Predicted", color=TEXT_MID)
        ax.set_ylabel("True", color=TEXT_MID)
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    fig.tight_layout()
    fig.savefig(os.path.join(out, f"fig_confusion_clean_{variant}.png"))
    plt.close(fig)


def fig_learning_curves(runs, out, variant="std"):
    """Validation accuracy per epoch on original and on noisy images, one line per seed, with the selected
    checkpoint marked. Shows how the selection score behaves for models trained without noise."""
    archs = [a for a in ARCHS if seeds_of(runs, a, variant)]
    if not archs:
        return
    from matplotlib.lines import Line2D
    fig, axes = plt.subplots(1, len(archs), figsize=(4.4 * len(archs), 4.1), squeeze=False)
    for ax, a in zip(axes[0], archs):
        c = ARCH_COLOR[a]
        for s_ in seeds_of(runs, a, variant):
            r = runs[(a, variant, s_)]
            ep = np.array([h["epoch"] for h in r["history"]])
            vc = np.array([h["val_acc_clean"] for h in r["history"]])
            vn = np.array([h["val_acc_noisy"] for h in r["history"]])
            ax.plot(ep, vc, color=c, linewidth=1.6, alpha=0.9)
            ax.plot(ep, vn, color=c, linewidth=1.6, alpha=0.9, linestyle=(0, (4, 2)))
            b = r["best_epoch"] - 1
            ax.plot([ep[b]], [vc[b]], marker="o", markersize=8, markerfacecolor=c, markeredgecolor=BG_COLOR,
                    markeredgewidth=1.5, linestyle="none", zorder=5)
            ax.plot([ep[b]], [vn[b]], marker="o", markersize=8, markerfacecolor=BG_COLOR, markeredgecolor=c,
                    markeredgewidth=2, linestyle="none", zorder=5)
        style(ax)
        ax.set_title(ARCH_LABEL[a])
        ax.set_xlabel("Epoch")
        ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True, nbins=8))
    axes[0][0].set_ylabel("Validation accuracy")
    handles = [Line2D([], [], color=TEXT_MID, linewidth=1.8, label="original validation images"),
               Line2D([], [], color=TEXT_MID, linewidth=1.8, linestyle=(0, (4, 2)), label="noisy validation images"),
               Line2D([], [], color=TEXT_MID, marker="o", markersize=8, markerfacecolor=TEXT_MID, markeredgecolor=BG_COLOR,
                      linestyle="none", label="selected checkpoint (one line per seed)")]
    fig.suptitle(f"Validation accuracy during {VAR_LABEL[variant]}", x=0.01, y=0.985, ha="left", va="top",
                 fontsize=11, fontweight="bold", color=TEXT_DARK)
    fig.legend(handles=handles, loc="upper left", ncol=3, frameon=False, fontsize=9, labelcolor=TEXT_MID,
               bbox_to_anchor=(0.0, 0.925), handlelength=3)
    fig.tight_layout(rect=(0, 0, 1, 0.85))
    fig.savefig(os.path.join(out, "fig_learning_curves.png" if variant == "std" else f"fig_learning_curves_{variant}.png"))
    plt.close(fig)


def main(out_dir):
    runs = load_runs(out_dir)
    if not runs:
        print("no final runs found in", out_dir)
        return
    out = os.path.join(out_dir, "analysis")
    os.makedirs(out, exist_ok=True)
    print(write_tables(runs, out))
    jobs = [("degradation accuracy", lambda: fig_degradation(runs, out, "acc", name="fig_degradation_acc")),
            ("degradation macro-F1", lambda: fig_degradation(runs, out, "macro_f1", name="fig_degradation_macro_f1")),
            ("relative accuracy", lambda: fig_degradation(runs, out, "acc", relative=True, name="fig_relative_acc")),
            ("extended sweep", lambda: fig_degradation(runs, out, "acc", extended=True, name="fig_degradation_acc_extended")),
            ("by architecture", lambda: fig_by_architecture(runs, out)),
            ("augmentation effect", lambda: fig_aug_delta(runs, out)),
            ("per-class recall", lambda: fig_recall_heatmaps(runs, out)),
            ("confusion std", lambda: fig_confusion(runs, out, "std")),
            ("confusion aug", lambda: fig_confusion(runs, out, "aug")),
            ("learning curves", lambda: fig_learning_curves(runs, out)),
            ("learning curves aug", lambda: fig_learning_curves(runs, out, "aug"))]
    for name, job in jobs:
        try:
            job()
        except Exception as e:  # continue if one plot fails
            print(f"figure '{name}' failed: {e!r}")
    print("analysis written to", out)


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
# Load and deduplicate FER-2013
import numpy as np
import ferlib as fl
importlib.reload(fl)

NPZ = os.path.join(OUT_DIR, "fer_dedup.npz")
if os.path.exists(NPZ):
    data = fl.load_data(NPZ)
else:
    csv_path = CSV_PATH
    if csv_path is None:
        from huggingface_hub import hf_hub_download
        try:
            tgz = hf_hub_download("DerrickUnleashed/FER-2013", "fer2013.tar.gz", repo_type="dataset")
            csv_path = os.path.join(tempfile.gettempdir(), "fer2013.csv")
            with tarfile.open(tgz) as tf:
                member = next(m for m in tf.getmembers() if m.isfile() and m.name.endswith("fer2013.csv"))
                with tf.extractfile(member) as src, open(csv_path, "wb") as dst:
                    shutil.copyfileobj(src, dst)
        except Exception as e:
            print("fer2013.tar.gz route failed:", repr(e), "-> using icml_face_data.csv (same images)")
            csv_path = hf_hub_download("DerrickUnleashed/FER-2013", "icml_face_data.csv", repo_type="dataset")
    X, y, usage = fl.read_fer_csv(csv_path)
    data, audit = fl.audit_and_dedup(X, y, usage)
    print(json.dumps(audit, indent=1))
    json.dump(audit, open(os.path.join(OUT_DIR, "data_audit.json"), "w"), indent=1)
    fl.save_data(data, NPZ)

sizes = {s: int(len(data[s][1])) for s in data}
for s in ("train", "val", "test"):
    print(f"{s:5s} n={sizes[s]:6d}  " + "  ".join(f"{n}={v}" for n, v in zip(fl.FER_LABELS, fl.class_counts(data[s][1]))))
assert sizes == fl.EXPECTED_SIZES, f"split sizes {sizes} differ from the proposal {fl.EXPECTED_SIZES}"
assert fl.data_fingerprint(data) == fl.EXPECTED_FINGERPRINT, "data fingerprint mismatch"
print("data verified; row order match:",
      fl.row_order_fingerprint(data) == fl.EXPECTED_ROW_ORDER_FINGERPRINT)

In [ ]:
# Run experiments
import analyze
importlib.reload(analyze)

device = torch.device("cuda")

# Quick smoke test on one batch per model
for arch in ARCHS:
    m = fl.build_model(arch).to(device)
    xb = fl.to_float(data["train"][0][:32]).to(device)
    yb = torch.from_numpy(data["train"][1][:32]).to(device)
    with torch.autocast("cuda"):
        loss = torch.nn.functional.cross_entropy(m(xb), yb)
    loss.backward()
    print(f"smoke test {arch}: ok ({fl.count_params(m):,} parameters, loss {loss.item():.3f})")
    del m, xb, yb, loss
torch.cuda.empty_cache()

runner = fl.Runner(OUT_DIR, data, device, partial_dir=PARTIAL_DIR, reuse_dir=REUSE_DIR)
t_start = time.time()
for arch in ARCHS:
    sel, results = runner.run_architecture(arch, seeds=SEEDS, **fl.ARCH_SETTINGS[arch])
    runner.log(f"[{arch}] done: lr={sel['lr']:g}, aug p={sel['aug_p']:g}, sigma_max={sel['aug_sigma_max']:g}; "
               f"elapsed {(time.time() - t_start) / 60:.0f} min")
    analyze.main(OUT_DIR)          # update summary tables and plots

In [ ]:
# Reference runs (clean-only validation checkpoint selection)
for arch in ARCHS:
    runner.run_reference(arch, seeds=SEEDS, **fl.ARCH_SETTINGS[arch])
    runner.log(f"[{arch}] reference runs done; elapsed {(time.time() - t_start) / 60:.0f} min")

In [ ]:
# Summary tables and plots
from IPython.display import Image, Markdown, display

analyze.main(OUT_DIR)
display(Markdown(open(os.path.join(OUT_DIR, "analysis", "results_summary.md")).read()))
for name in ("fig_degradation_acc", "fig_relative_acc", "fig_by_architecture", "fig_augmentation_effect",
             "fig_per_class_recall", "fig_confusion_clean_std", "fig_learning_curves"):
    p = os.path.join(OUT_DIR, "analysis", name + ".png")
    if os.path.exists(p):
        display(Image(filename=p, width=900))

In [ ]:
# Package results into zip
bundle = os.path.join(OUT_DIR, "results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ("final/*.json", "final/*_preds.npz", "tune/*.json", "selection_*.json", "analysis/*",
                    "data_audit.json", "log.txt"):
        for p in sorted(glob.glob(os.path.join(OUT_DIR, pattern))):
            z.write(p, os.path.relpath(p, OUT_DIR))
print(f"bundle: {bundle} ({os.path.getsize(bundle) / 1e6:.1f} MB)")
n_final = len(glob.glob(os.path.join(OUT_DIR, "final", "*.json")))
print(f"final runs finished: {n_final} / {len(ARCHS) * 3 * len(SEEDS)}")
if IN_COLAB:
    try:
        from google.colab import files
        files.download(bundle)
    except Exception as e:
        print(f"automatic download skipped ({e!r}); bundle saved to {bundle}")